# Tensor Indexing Memo

This memo formalizes tensor indexing using mathematical first principles. Brief outline
1. Establish the mental models of treating tensors as family of math functions.
2. Frame tensor indexing as function compositions.

## Tensor as family of math functions

A tensor is an n-dimensional array of primitive types. Common types are `torch.int16`, `torch.int32`, `torch.float16`, `torch.float32`, `torch.bool`. One can think of a tensor as a function mapping from n-dimensional integer indexes to a primitive value, e.g.:
* `t1: Float[torch.Tensor, 'batch seq d_model']` can be modeled as $t1: \mathbb{N_0}^3 \to \mathbb{R}$
* `t2: Bool[torch.Tensor, 'batch seq']` can be modeled as $t2: \mathbb{N_0}^2 \to \{0, 1\}$

But the codomain isn't limited to just primitives. In fact, codomains can be tensor functions too:
* `t1: Float[torch.Tensor, 'batch seq d_model']` can be modeled as $t1: \mathbb{N_0}^2 \to \mathbb{N_0} \times \mathbb{R}$, or as $t1: \mathbb{N_0} \to \mathbb{N_0}^2 \times \mathbb{R}$
* `t2: Bool[torch.Tensor, 'batch seq']` can be modeled as $t2: \mathbb{N_0} \to \mathbb{N_0} \times \{0, 1\}$

So a tensor type is really a family of math functions. And just like math functions, tensors can be "evaluated" by indexing at specific points to both (a) pick a math function from the family, and (b) evaluate that function at the given fixed point. For example, `t1[1, 0, 6]` means pick the math function $t1: \mathbb{N_0}^3 \to \mathbb{R}$, and plug in `(1, 0, 6)` to evaluate that function.

## Tensor as index - function composition

The function domain of a tensor ($\mathbb{N_0}^n$) is in itself a valid n-dimensional array of primitive type (in this case, `torch.int32`). Thus, The function domain can be though of as a tensor as well. This means we can perform function composition over a sequence of tensor functions, with the caveat that the tensor being used as an index *must*:
* have $\mathbb{N_0}^n$ as its domain, i.e. must be of type `Int[torch.Tensor, '...']`;
* the int values in the index tensor must be within range of the 0-th axis of the source tensor being indexed into;

Note the 2nd bullet point is subtle - why 0-th axis? Because when we write `source[index]`, by convention the `[.]` operator operates against the 0-th axis. This also implies the remaining axes of `source` are considerd as *output* of the indexing operation, aka *codomain* of the tensor after function composition. This echos the previous cell's notion that a tensor is a *family* of math functions, and in this case we happen to pick the family member $\mathbb{N_0} \to \mathbb{N_0}^{n-1} \times \mathbb{R}$.

Putting everything together: when we write `source[index]` in Python, mathematically this is really just a function composition:
* `source` tensor's function form is $\mathbb{N_0} \to \mathbb{N_0}^{n-1} \times \mathbb{R}$;
* `index` tensor's function form is $target: \mathbb{N_0}^m \to \mathbb{N_0}$;
* `source[index]` tensor's function form is $source \circ target$, which reduces to $\mathbb{N_0}^m \to \mathbb{N}^{n-1} \times \mathbb{R}$

This also imples `source[index].shape == (*index.shape, *source[1:].shape)`. There are no restrictions to shape of index or source. Only restrictions are the caveats noted above: index must be valued as int, and int values must be within range of source's 0-th axis.

In [1]:
import torch

# source.shape == (2, 3)
# source: N -> N x R
source = torch.Tensor([
    [1, 2, 3],
    [4, 5, 6],
]).to(torch.float32)

# index.shape == (2, 2, 3)
# index: N^3 -> N
index = torch.Tensor([
    # values must be within source's 0th axis values, i.e. < 2.
    [
        [1, 0, 1],
        [1, 1, 1],
    ],
    [
        [0, 0, 1],
        [0, 1, 1],
    ],
]).to(torch.int32)  # index tensor must be int.

# fuction composition: source[index].shape == (2, 2, 3, 3)
# source o index: N^3 -> R
assert source[index].shape == (2, 2, 3, 3)
expected = torch.Tensor([
    [
        [
            # index [1, 0, 1] -> pick source's 1st row, then 0th row, then 1st row
            [4, 5, 6],
            [1, 2, 3],
            [4, 5, 6],
        ],
        [
            # index [1, 1, 1] -> pick source's 1st row, then 1th row, then 1st row
            [4, 5, 6],
            [4, 5, 6],
            [4, 5, 6],
        ],
    ],
    [
        [
            # index [0, 0, 1] -> pick source's 0th row, then 0th row, then 1st row
            [1, 2, 3],
            [1, 2, 3],
            [4, 5, 6],
        ],
        [
            # index [0, 1, 1] -> pick source's 0th row, then 1th row, then 1st row
            [1, 2, 3],
            [4, 5, 6],
            [4, 5, 6],
        ],
    ],
]).to(torch.float32)
assert torch.equal(source[index], expected)

## PyTorch Tensor Indexing API: torch.gather

The `torch.gather` API has the following signature. Note this isn't exactly what's appearing in the official doc https://docs.pytorch.org/docs/2.13/generated/torch.gather.html (`torch.gather(input, dim, index, *, sparse_grad=False, out=None) -> Tensor`), but hopefully this gives a clearer mental model:

```python
def gather(
    input: Float[torch.Tensor, 'a b c'],
    dim: int,  # which dimension of **input** to gather
    index: Float[torch.Tensor, 'd e f'],  # num dims must match input
    ...) -> Float[torch.Tensor, 'd e f']  # shape must match index
    ...
```

Note the following invariant from `gather`'s API contract:
1. `len(input.shape) == len(index.shape)`, though it's ok to have `input.shape != index.shape` (e.g. index[i] may be 1 to leverage broadcast).
2. `index.shape[d] in (input.shape[d], 1)` for all `d != dim` (1 is allowed for broadcast support). This is because for `d == dim`, `index` is free to select any number of elements from `dim`-th dimension in `input`.
3. `index.shape == output.shape`; this must be exact match.

Offficial doc's example gives additional intuition:

```python
out[i][j][k] = input[index[i][j][k]][j][k]  # if dim == 0
out[i][j][k] = input[i][index[i][j][k]][k]  # if dim == 1
out[i][j][k] = input[i][j][index[i][j][k]]  # if dim == 2
```

In [18]:
input = torch.tensor([
    [1, 5, 3],
    [4, 2, 6],
])
index = torch.tensor([
    [1, 0, 1],
    [0, 1, 0],
])
# gather at dim=0 => output[i][j] == input[index[i][j]][j]
actual = torch.gather(input=input, dim=0, index=index)
expected = torch.tensor([
    [4, 5, 6],
    [1, 2, 3],
])
assert torch.equal(expected, actual), f'gather output: {actual}'
# gather at dim=1 => output[i][j] == input[i][index[i][j]]
actual = torch.gather(input=input, dim=1, index=index)
expected = torch.tensor([
    [5, 1, 5],
    [4, 2, 4],
])
assert torch.equal(expected, actual), f'gather output: {actual}'

### Case study: cross_entropy

The transformer language model finally outputs a single tensor of logits, having shape `(batch, seq, vocab)`. During training, the target sequence has shape `(batch, seq)`, whose value is an integer representing the vocab id. This is a good use case for the `gather` API, specifically to find the i-th entry in each logit tensor where i matches the target's vocab index.

In [ ]:
B, S, V = 1, 2, 3
logits = torch.tensor([
    # batch 0
    [
        # seq 0 - these are logit scores, not vocab indexes.
        [
            # vocab 0
            100,
            # vocab 1
            101,
            # vocab 2
            99999,
        ],
        # seq 1
        [
            # vocab 0
            1,
            # vocab 1
            201,
            # vocab 2
            200,
        ],
    ],
])
assert logits.shape == (B, S, V)
index = torch.tensor([
    # batch 0
    [
        # seq 0 - these are vocab indexes, not scores.
        2,  # good - matches high score
        # seq 
        0,  # bad - matches low score
    ],
])
assert index.shape == (B, S)

actual = torch.gather(
    input=logits,
    dim=-1,
    index=index.unsqueeze(-1),
).squeeze(-1)
assert actual.shape == (B, S)
expected = torch.tensor([
    [
        99999,
        1,
    ],
])
assert torch.equal(expected, actual)

RuntimeError: Index tensor must have the same number of dimensions as input tensor